In [5]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

!curl -O https://raw.githubusercontent.com/yszanwar/phase2_qrt_challenge/refs/heads/main/scripts/utils.py

import utils
importlib.reload(utils)

from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go

import warnings
warnings.filterwarnings("ignore")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 18007  100 18007    0     0  71697      0 --:--:-- --:--:-- --:--:-- 72028


In [1]:
from io import BytesIO
import requests
import pandas as pd
from tqdm import tqdm
import ast

def download_with_progress(url):
    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    buffer = BytesIO()
    with tqdm(total=total_size, unit='B', unit_scale=True, desc=url.split("/")[-1]) as pbar:
        for chunk in response.iter_content(chunk_size=1024 * 1024):  # 1MB chunks
            if chunk:
                buffer.write(chunk)
                pbar.update(len(chunk))

    buffer.seek(0)
    return buffer

# -------- Features --------
url_features = "https://github.com/yszanwar/phase2_qrt_challenge/releases/download/features/features.parquet"
features = pd.read_parquet(download_with_progress(url_features), engine="pyarrow")

# -------- Universe --------
url_universe_1m = "https://github.com/yszanwar/phase2_qrt_challenge/releases/download/dataset/universe_1m.parquet"
universe = pd.read_parquet(download_with_progress(url_universe_1m), engine="pyarrow")

# -------- Returns --------
url_returns = "https://github.com/yszanwar/phase2_qrt_challenge/releases/download/dataset/returns.parquet"
returns = pd.read_parquet(download_with_progress(url_returns), engine="pyarrow")


features.columns = pd.MultiIndex.from_tuples(
    [ast.literal_eval(col) for col in features.columns]
)
features.index = universe.index

sector_mapping = pd.read_csv("https://raw.githubusercontent.com/yszanwar/phase2_qrt_challenge/refs/heads/main/top_5000_us_by_marketcap.csv").set_index("symbol").sector


features.parquet: 100%|██████████| 1.88G/1.88G [00:11<00:00, 163MB/s]
universe_1m.parquet: 100%|██████████| 3.31M/3.31M [00:00<00:00, 75.0MB/s]
returns.parquet: 100%|██████████| 124M/124M [00:00<00:00, 249MB/s]


In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, "stores")

features = pd.read_parquet(os.path.join(DATA_DIR, "features.parquet"))

universe = pd.read_parquet(os.path.join(DATA_DIR, "universe_1m.parquet"))

returns = pd.read_parquet(os.path.join(DATA_DIR, "returns.parquet"))

In [ ]:
sector_mapping = pd.read_csv(os.path.join(BASE_DIR, "top_5000_us_by_marketcap.csv")).set_index("symbol").sector

In [6]:
# Testing the backtest_portfolio function for a specific signal

def generate_portfolio_vectorized(
    entire_features: pd.DataFrame,
    universe: pd.DataFrame,
    start_date: str,
    end_date: str,
    signal_column: str,
    neutralize_by: str = None,
    sector_mapping: pd.DataFrame = None
):
    # Validate date format
    try:
        start_dt = datetime.datetime.strptime(start_date, '%Y-%m-%d')
        end_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d')
        cutoff_date = datetime.datetime.strptime('2005-01-01', '%Y-%m-%d')
    except ValueError:
        raise ValueError("start_date and end_date must be strings in 'YYYY-MM-DD' format.")

    # Ensure start_date is before end_date
    if start_dt >= end_dt:
        raise ValueError("start_date must be earlier than end_date.")

    # Ensure start_date is not before '2005-01-01'
    if start_dt < cutoff_date:
        raise ValueError("start_date must be later than '2005-01-01'.")

    # Get trading days within the date range
    trading_days = universe.index[(universe.index >= start_dt) & (universe.index <= end_dt)]

    if len(trading_days) == 0:
        raise ValueError("No Trading Days in the specified dates")

    portfolio = 0

    universe_boolean = universe.loc[:end_date].astype(bool)

    features_ = entire_features.loc[:end_date]

    signal1 = features_[signal_column].shift(5)
    signal1 = signal1.where(universe_boolean, np.nan)
    signal1 = signal1.rank(axis=1, method="min", ascending=True)
    if neutralize_by == "market":
        signal1 = signal1.sub(signal1.mean(axis=1), axis=0)
        signal1 = signal1.sub(signal1.mean(axis=1), axis=0)
        signal1 = signal1.div(signal1.abs().sum(axis=1), axis=0)
        signal1 = signal1.fillna(0)
    if neutralize_by == "sector" and sector_mapping is not None:
        sector_mapping = sector_mapping.reindex(signal1.columns, fill_value="Others")
        sector_mapping = sector_mapping.loc[signal1.columns]
        signal1 = signal1.sub(signal1.groupby(sector_mapping, axis=1).transform("mean"), axis=0)
        signal1 = signal1.div(signal1.abs().sum(axis=1), axis=0)
        signal1 = signal1.fillna(0)

    portfolio = -1 * signal1

    portfolio = portfolio.div(portfolio.abs().sum(axis=1), axis=0)

    return portfolio.fillna(0).loc[start_date:end_date]

In [23]:
benchmark_portfolio_vectorized = generate_portfolio_vectorized(
    features,
    universe,
    "2010-01-01",
    "2026-01-01",
    "accumulation_distribution_index",
    "sector",
    sector_mapping
)

In [24]:
benchmark_portfolio_vectorized

,AAPL,ABBV,ABT,ADI,AMAT,AMD,AMGN,AMZN,ANET,APH,...,VEEA,VNCE,VNRX,VNTG,WKHS,WYHG,YSXT,YXT,ZDAI,ZKIN
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-06,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2010-01-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,-0.001111,-0.001280,-0.001323,-0.000958,-0.001040,-0.001098,-0.001208,-0.001191,-0.000850,-0.001059,...,-0.0,-0.0,0.000543,-0.0,-0.0,-0.0,0.000304,-0.0,-0.0,-0.0
2025-12-26,-0.001121,-0.001292,-0.001334,-0.000965,-0.001047,-0.001108,-0.001219,-0.001209,-0.000850,-0.001068,...,-0.0,-0.0,0.000542,-0.0,-0.0,-0.0,0.000296,-0.0,-0.0,-0.0
2025-12-29,-0.001111,-0.001295,-0.001335,-0.000956,-0.001037,-0.001098,-0.001221,-0.001199,-0.000852,-0.001060,...,-0.0,-0.0,0.000535,-0.0,-0.0,-0.0,0.000302,-0.0,-0.0,-0.0


In [26]:
sr_vectorized, pnl_vectorized = backtest_portfolio(benchmark_portfolio_vectorized.loc["2010":"2019"], returns.loc["2010":"2019"], universe.loc["2010":"2019"], True, True)

Gross Sharpe Ratio:  0.636
Net Sharpe Ratio:  0.606
Turnover %:  4.391


In [17]:
feature_results = pd.DataFrame(columns=["feature", "sharpe_ratio"])
feature_pnls = {}
for col in features.columns.get_level_values(0).unique():
    print(f"Testing feature: {col}")
    portfolio_vectorized = generate_portfolio_vectorized(
        features,
        universe,
        "2010-01-01",
        "2026-01-01",
        col,
        "sector",
        sector_mapping
    )
    sr_vectorized, pnl_vectorized = backtest_portfolio(portfolio_vectorized.loc["2010":"2019"], returns.loc["2010":"2019"], universe.loc["2010":"2019"], False, False)
    new_row = pd.DataFrame([{
        "feature": col,
        "sharpe_ratio": sr_vectorized
    }])
    feature_pnls[col] = pnl_vectorized

    feature_results = pd.concat([feature_results, new_row], ignore_index=True)

Testing feature: relative_strength_index
Testing feature: williams_r
Testing feature: rsi
Testing feature: volatility_20
Testing feature: volatility_60
Testing feature: trend_1_3
Testing feature: trend_5_20
Testing feature: trend_20_60
Testing feature: average_true_range
Testing feature: macd
Testing feature: macd_signal
Testing feature: macd_histogram
Testing feature: trix
Testing feature: commodity_channel_index
Testing feature: chande_momentum_oscillator
Testing feature: ichimoku_conversion
Testing feature: ichimoku_base
Testing feature: ichimoku_leading_a
Testing feature: ichimoku_leading_b
Testing feature: know_sure_thing
Testing feature: ultimate_oscillator
Testing feature: aroon_up
Testing feature: aroon_down
Testing feature: aroon_oscillator
Testing feature: stochastic_k
Testing feature: stochastic_d
Testing feature: on_balance_volume
Testing feature: ease_of_movement
Testing feature: chaikin_money_flow
Testing feature: accumulation_distribution_index
Testing feature: volume


In [19]:
top_signals = feature_results.sort_values(by="sharpe_ratio", ascending=False)
top_signals

,feature,sharpe_ratio
8,average_true_range,2.298
15,ichimoku_conversion,1.655
16,ichimoku_base,1.641
17,ichimoku_leading_a,1.640
18,ichimoku_leading_b,1.636
29,accumulation_distribution_index,0.606
9,macd,0.528
25,stochastic_d,0.479
10,macd_signal,0.453
13,commodity_channel_index,0.414


In [20]:
selected_signals = pd.DataFrame(feature_pnls).loc[:, ['ichimoku_conversion', 'accumulation_distribution_index', 'average_true_range']]

In [21]:
selected_signals.corr()

,ichimoku_conversion,accumulation_distribution_index,average_true_range
ichimoku_conversion,1.000000,0.857898,0.671579
accumulation_distribution_index,0.857898,1.000000,0.420223
average_true_range,0.671579,0.420223,1.000000


In [22]:
selected_signals.to_parquet("selected_signals.parquet")